In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mfekadu/darpa-timit-acousticphonetic-continuous-speech
!unzip darpa-timit-acousticphonetic-continuous-speech.zip

Показано результат, скорочений до останніх рядків (5000).
  inflating: data/TRAIN/DR6/MTXS0/SA1.WAV  
  inflating: data/TRAIN/DR6/MTXS0/SA1.WAV.wav  
  inflating: data/TRAIN/DR6/MTXS0/SA1.WRD  
  inflating: data/TRAIN/DR6/MTXS0/SA2.PHN  
  inflating: data/TRAIN/DR6/MTXS0/SA2.TXT  
  inflating: data/TRAIN/DR6/MTXS0/SA2.WAV  
  inflating: data/TRAIN/DR6/MTXS0/SA2.WAV.wav  
  inflating: data/TRAIN/DR6/MTXS0/SA2.WRD  
  inflating: data/TRAIN/DR6/MTXS0/SI1060.PHN  
  inflating: data/TRAIN/DR6/MTXS0/SI1060.TXT  
  inflating: data/TRAIN/DR6/MTXS0/SI1060.WAV  
  inflating: data/TRAIN/DR6/MTXS0/SI1060.WAV.wav  
  inflating: data/TRAIN/DR6/MTXS0/SI1060.WRD  
  inflating: data/TRAIN/DR6/MTXS0/SI1690.PHN  
  inflating: data/TRAIN/DR6/MTXS0/SI1690.TXT  
  inflating: data/TRAIN/DR6/MTXS0/SI1690.WAV  
  inflating: data/TRAIN/DR6/MTXS0/SI1690.WAV.wav  
  inflating: data/TRAIN/DR6/MTXS0/SI1690.WRD  
  inflating: data/TRAIN/DR6/MTXS0/SI2320.PHN  
  inflating: data/TRAIN/DR6/MTXS0/SI2320.TXT  
  inflatin

In [ ]:
import librosa

import numpy as np
import os
import matplotlib.pyplot as plt
import math
import random
import pandas as pd
import soundfile as sf
from pathlib import Path
from typing import Dict, List, Tuple
from scipy import signal
from scipy.signal import butter, filtfilt, wiener
from scipy.ndimage import median_filter
from scipy.fft import rfft, irfft

In [ ]:
TIMIT_ROOT = Path("data")
OUTPUT_ROOT = Path("TIMIT_enhanced")
BENCHMARK_CSV = Path("results.csv")

TARGET_SR = 16000
AUDIO_EXTENSIONS = {".wav", ".WAV"}


HPF_CUTOFF = 80.0
LPF_CUTOFF = 4000.0


N_FFT = 512
HOP = 128
WIN_LEN = 512
NOISE_SEC = 0.20
GATE_STD_MULT = 1.5
FLOOR_GAIN = 0.15


def list_wav_files(root: Path) -> List[Path]:
    files = []
    for p in root.rglob("*"):
        if p.suffix in AUDIO_EXTENSIONS:
            files.append(p)
    return sorted(files)


def load_wav(path: Path) -> Tuple[np.ndarray, int]:
    x, sr = sf.read(path)
    if x.ndim > 1:
        x = np.mean(x, axis=1)
    x = x.astype(np.float32)
    return x, sr


def rms(x: np.ndarray, eps: float = 1e-10) -> float:
    return float(np.sqrt(np.mean(x ** 2) + eps))


def peak_normalize(x: np.ndarray, peak: float = 0.98) -> np.ndarray:
    m = np.max(np.abs(x))
    if m < 1e-10:
        return x.copy()
    return (x / m) * peak


def remove_dc(x: np.ndarray) -> np.ndarray:
    return x - np.mean(x)


def pre_emphasis(x: np.ndarray, coeff: float = 0.97) -> np.ndarray:
    y = np.empty_like(x)
    y[0] = x[0]
    y[1:] = x[1:] - coeff * x[:-1]
    return y


def de_emphasis(x: np.ndarray, coeff: float = 0.97) -> np.ndarray:
    y = np.empty_like(x)
    y[0] = x[0]
    for i in range(1, len(x)):
        y[i] = x[i] + coeff * y[i - 1]
    return y


# filters
def butter_highpass(x: np.ndarray, sr: int, cutoff: float = HPF_CUTOFF, order: int = 4) -> np.ndarray:
    b, a = signal.butter(order, cutoff / (sr / 2), btype="highpass")
    return signal.filtfilt(b, a, x).astype(np.float32)


def butter_lowpass(x: np.ndarray, sr: int, cutoff: float = LPF_CUTOFF, order: int = 4) -> np.ndarray:
    b, a = signal.butter(order, cutoff / (sr / 2), btype="lowpass")
    return signal.filtfilt(b, a, x).astype(np.float32)


def notch_filter(x: np.ndarray, sr: int, freq: float = 50.0, q: float = 30.0) -> np.ndarray:
    if freq >= sr / 2:
        return x
    b, a = signal.iirnotch(freq / (sr / 2), q)
    return signal.filtfilt(b, a, x).astype(np.float32)


def speech_bandpass_pipeline(x: np.ndarray, sr: int) -> np.ndarray:
    x = butter_highpass(x, sr, HPF_CUTOFF)
    x = butter_lowpass(x, sr, LPF_CUTOFF)
    return x


def adaptive_wiener(x: np.ndarray, mysize: int = 29) -> np.ndarray:
    return wiener(x, mysize=mysize).astype(np.float32)


# STFT / ISTFT

def stft_mag_phase(x: np.ndarray, n_fft: int = N_FFT, hop: int = HOP, win_len: int = WIN_LEN):
    f, t, Zxx = signal.stft(
        x,
        fs=1.0,
        window="hann",
        nperseg=win_len,
        noverlap=win_len - hop,
        nfft=n_fft,
        boundary="zeros",
        padded=True,
    )
    mag = np.abs(Zxx)
    phase = np.angle(Zxx)
    return mag, phase


def istft_from_mag_phase(mag: np.ndarray, phase: np.ndarray, hop: int = HOP, win_len: int = WIN_LEN) -> np.ndarray:
    Zxx = mag * np.exp(1j * phase)
    _, x = signal.istft(
        Zxx,
        fs=1.0,
        window="hann",
        nperseg=win_len,
        noverlap=win_len - hop,
        input_onesided=True,
        boundary=True,
    )
    return x.astype(np.float32)


def estimate_noise_frames(x: np.ndarray, sr: int, noise_sec: float = NOISE_SEC) -> np.ndarray:
    n = max(1, int(noise_sec * sr))
    if len(x) < 2 * n:
        noise = x[:n]
    else:
        noise = np.concatenate([x[:n], x[-n:]])
    return noise.astype(np.float32)


def spectral_gate_denoise(
    x: np.ndarray,
    sr: int,
    n_fft: int = N_FFT,
    hop: int = HOP,
    win_len: int = WIN_LEN,
    gate_std_mult: float = GATE_STD_MULT,
    floor_gain: float = FLOOR_GAIN,
) -> np.ndarray:
    noise = estimate_noise_frames(x, sr)
    noise_mag, _ = stft_mag_phase(noise, n_fft=n_fft, hop=hop, win_len=win_len)

    noise_mean = np.mean(noise_mag, axis=1, keepdims=True)
    noise_std = np.std(noise_mag, axis=1, keepdims=True)
    threshold = noise_mean + gate_std_mult * noise_std

    mag, phase = stft_mag_phase(x, n_fft=n_fft, hop=hop, win_len=win_len)

    mask = mag >= threshold
    soft_mask = np.where(mask, 1.0, floor_gain)

    soft_mask = median_filter(soft_mask, size=(5, 3))

    out_mag = mag * soft_mask
    y = istft_from_mag_phase(out_mag, phase, hop=hop, win_len=win_len)

    if len(y) < len(x):
        y = np.pad(y, (0, len(x) - len(y)))
    elif len(y) > len(x):
        y = y[:len(x)]

    return y.astype(np.float32)


def normalize_rms(x: np.ndarray, target_db: float = -22.0) -> np.ndarray:
    target_rms = 10 ** (target_db / 20.0)
    current_rms = rms(x)
    if current_rms < 1e-10:
        return x.copy()
    gain = target_rms / current_rms
    y = x * gain
    return peak_normalize(y, peak=0.98)

def enhance_audio(x: np.ndarray, sr: int) -> np.ndarray:
    """
    Enhancement pipeline:
    1) remove DC
    2) notch at 50 Hz and harmonics
    3) speech band filtering
    4) Wiener filtering
    5) spectral gating
    6) RMS normalization
    """
    y = x.copy()

    y = remove_dc(y)

    for freq in [50, 100, 150]:
        if freq < sr / 2:
            y = notch_filter(y, sr, freq=freq, q=30.0)


    y = speech_bandpass_pipeline(y, sr)

    y = adaptive_wiener(y, mysize=29)

    y = pre_emphasis(y)

    y = spectral_gate_denoise(y, sr)

    y = de_emphasis(y)

    y = normalize_rms(y, target_db=-22.0)
    y = peak_normalize(y, peak=0.98)

    return y.astype(np.float32)

# metrics

def spectral_centroid(x, sr):

    freqs, times, S = signal.spectrogram(x, sr)

    centroid = np.sum(freqs[:, None] * S, axis=0) / (np.sum(S, axis=0) + 1e-10)

    return np.mean(centroid)


def log_spectral_distance(x, y):
    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    _, _, X = signal.stft(x, nperseg=256)
    _, _, Y = signal.stft(y, nperseg=256)

    logX = np.log(np.abs(X)**2 + 1e-8)
    logY = np.log(np.abs(Y)**2 + 1e-8)

    dist = np.sqrt(np.mean((logX - logY)**2))

    return float(dist)

def snr(original: np.ndarray, processed: np.ndarray) -> float:

   # snr between original and processed signal.

    n = min(len(original), len(processed))
    original = original[:n]
    processed = processed[:n]

    noise = original - processed

    signal_power = np.mean(original ** 2)
    noise_power = np.mean(noise ** 2) + 1e-10

    snr_value = 10 * np.log10(signal_power / noise_power)

    return float(snr_value)

def pipeline_baseline(x, sr):
    y = remove_dc(x)
    y = normalize_rms(y)
    return peak_normalize(y)

def pipeline_bandpass(x, sr):
    y = remove_dc(x)
    y = speech_bandpass_pipeline(y, sr)
    y = normalize_rms(y)
    return peak_normalize(y)

def pipeline_wiener(x, sr):
    y = remove_dc(x)
    y = speech_bandpass_pipeline(y, sr)
    y = adaptive_wiener(y, mysize=29)
    y = normalize_rms(y)
    return peak_normalize(y)

def pipeline_bandpass_wiener(x, sr):
    y = remove_dc(x)
    y = speech_bandpass_pipeline(y, sr)
    y = adaptive_wiener(y, mysize=29)
    y = normalize_rms(y)
    y = peak_normalize(y)
    return y.astype(np.float32)


def pipeline_full(x, sr):
    return enhance_audio(x, sr)

def save_audio(path, audio, sr):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, audio, sr)

def evaluate_dataset(timit_root, pipelines=None):

    if pipelines is None:
        pipelines = {
            "base": pipeline_baseline,
            "bandpass": pipeline_bandpass,
            "bandpass_wiener": pipeline_bandpass_wiener,
            "full": pipeline_full,
        }

    rows = []
    wav_files = list_wav_files(Path(timit_root))
    total = len(wav_files)

    for i, wav in enumerate(wav_files, 1):
        original, sr = load_wav(wav)

        rms_orig = rms(original)
        centroid_orig = spectral_centroid(original, sr)

        for pipeline_name, pipeline_fn in pipelines.items():
            processed = pipeline_fn(original, sr)
            out_path = OUTPUT_ROOT / pipeline_name / wav.relative_to(timit_root)
            save_audio(out_path, processed, sr)
            row = {
                "file": str(wav),
                "pipeline": pipeline_name,
                "sr": sr,

                "rms_original": rms_orig,
                "rms_processed": rms(processed),

                "centroid_original": centroid_orig,
                "centroid_processed": spectral_centroid(processed, sr),
                "centroid_delta": spectral_centroid(processed, sr) - centroid_orig,

                "lsd": log_spectral_distance(original, processed),

                "snr": snr(original, processed)
            }

            rows.append(row)

        if i % 100 == 0 or i == total:
            print(f"[{i}/{total}] processed")

    df_details = pd.DataFrame(rows)

    df_summary = (
        df_details
        .groupby("pipeline", as_index=False)
        .agg(
            n_files=("file", "count"),
            mean_rms_original=("rms_original", "mean"),
            mean_rms_processed=("rms_processed", "mean"),
            mean_centroid_original=("centroid_original", "mean"),
            mean_centroid_processed=("centroid_processed", "mean"),
            mean_centroid_delta=("centroid_delta", "mean"),
            mean_lsd=("lsd", "mean"),
            median_lsd=("lsd", "median"),
            std_lsd=("lsd", "std"),
            mean_snr=("snr", "mean"),
            median_snr=("snr", "median"),
        )
        .sort_values("mean_lsd")
        .reset_index(drop=True)
    )

    return df_details, df_summary

df_details, df_summary = evaluate_dataset(TIMIT_ROOT)
print(df_summary)
df_details.head()

[100/12600] processed
[200/12600] processed
[300/12600] processed
[400/12600] processed
[500/12600] processed
[600/12600] processed
[700/12600] processed
[800/12600] processed
[900/12600] processed
[1000/12600] processed
[1100/12600] processed
[1200/12600] processed
[1300/12600] processed
[1400/12600] processed
[1500/12600] processed
[1600/12600] processed
[1700/12600] processed
[1800/12600] processed
[1900/12600] processed
[2000/12600] processed
[2100/12600] processed
[2200/12600] processed
[2300/12600] processed
[2400/12600] processed
[2500/12600] processed
[2600/12600] processed
[2700/12600] processed
[2800/12600] processed
[2900/12600] processed
[3000/12600] processed
[3100/12600] processed
[3200/12600] processed
[3300/12600] processed
[3400/12600] processed
[3500/12600] processed
[3600/12600] processed
[3700/12600] processed
[3800/12600] processed
[3900/12600] processed
[4000/12600] processed
[4100/12600] processed
[4200/12600] processed
[4300/12600] processed
[4400/12600] process

,file,pipeline,sr,rms_original,rms_processed,centroid_original,centroid_processed,centroid_delta,lsd,snr
0,data/TEST/DR1/FAKS0/SA1.WAV,base,16000,0.024085,0.100317,1642.545099,1738.400901,95.855802,1.803387,-10.007786
1,data/TEST/DR1/FAKS0/SA1.WAV,bandpass,16000,0.024085,0.097389,1642.545099,1271.167380,-371.377718,1.736961,-9.754972
2,data/TEST/DR1/FAKS0/SA1.WAV,bandpass_wiener,16000,0.024085,0.081539,1642.545099,588.236466,-1054.308633,1.680600,-8.021189
3,data/TEST/DR1/FAKS0/SA1.WAV,full,16000,0.024085,0.081539,1642.545099,763.421185,-879.123913,1.689338,-8.021349
4,data/TEST/DR1/FAKS0/SA1.WAV.wav,base,16000,0.024085,0.100317,1642.545099,1738.400901,95.855802,1.803387,-10.007786
